# Notebook 2: Diabetes Prediction — Model Comparison & Evaluation

**BMI 503/511 — Machine Learning: Classification Methods**  
**Instructor:** Pratik Dutta, Ph.D. | Stony Brook University

---

## Learning Objectives
1. Work with a real clinical dataset (Pima Indians Diabetes)
2. Compare **all 5 classifiers** head-to-head
3. Plot and interpret **ROC curves** and **AUC scores**
4. Understand **cross-validation** for robust evaluation
5. Perform basic **hyperparameter tuning** with GridSearchCV

**Estimated time: ~35 minutes**

## 1. Load the Pima Indians Diabetes Dataset

- 768 female patients, 8 clinical features
- **Target:** 1 = Diabetic, 0 = Non-diabetic
- Features: Pregnancies, Glucose, Blood Pressure, Skin Thickness, Insulin, BMI, Diabetes Pedigree, Age

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, RocCurveDisplay)
import warnings
warnings.filterwarnings('ignore')

# Load dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Outcome'].value_counts().rename({0: 'Non-diabetic', 1: 'Diabetic'}))
print(f"\nNote: This is an IMBALANCED dataset ({df['Outcome'].mean():.1%} positive)")
df.describe().round(1)

### Data Cleaning

Some features have 0 values that are actually missing (e.g., Glucose=0 is impossible).

In [ ]:
# Replace physiologically impossible zeros with NaN, then impute with median
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in zero_cols:
    n_zeros = (df[col] == 0).sum()
    if n_zeros > 0:
        print(f"{col}: {n_zeros} zero values → replacing with median")
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())

print("\n✅ Data cleaning complete.")

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Which features correlate most strongly with Outcome (diabetes)?")

## 2. Prepare Data

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

## 3. Train All 5 Classifiers

We compare: Logistic Regression, Decision Tree, Random Forest, SVM, KNN

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

# All models use scaled data for fair comparison
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred,
                                target_names=['Non-diabetic', 'Diabetic']))

## 4. ROC Curves — The Gold Standard Comparison

The ROC curve plots **True Positive Rate** vs **False Positive Rate** at all thresholds.  
**AUC** (Area Under Curve) summarizes overall discriminative ability: 1.0 = perfect, 0.5 = random.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['#3B82F6', '#10B981', '#8B5CF6', '#EF4444', '#F59E0B']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All 5 Classifiers', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Discussion:")
print("   - Which classifier has the best AUC?")
print("   - Does the best AUC always mean the best choice for clinical use?")
print("   - What if we care more about catching all diabetic patients (high recall)?")

## 5. Cross-Validation — More Robust Evaluation

A single train/test split can be lucky or unlucky.  
**K-fold cross-validation** splits data into K folds and evaluates K times.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
print(f"{'Model':<25} {'Mean Accuracy':>15} {'Std':>8} {'Mean F1':>10}")
print('-' * 60)

for name, model_dict in results.items():
    # Re-create fresh model for CV
    model = type(model_dict['model'])(**model_dict['model'].get_params())

    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='f1')

    cv_results.append({
        'Model': name,
        'Mean Accuracy': acc_scores.mean(),
        'Std': acc_scores.std(),
        'Mean F1': f1_scores.mean(),
    })
    print(f"{name:<25} {acc_scores.mean():>15.4f} {acc_scores.std():>8.4f} {f1_scores.mean():>10.4f}")

print("\n💡 Cross-validation gives us confidence intervals, not just point estimates.")
print("   A model with high mean but high std may be unstable.")

In [ ]:
# Visualize CV results
cv_df = pd.DataFrame(cv_results)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(cv_df))
bars = ax.bar(x, cv_df['Mean Accuracy'], yerr=cv_df['Std'],
              color=colors, capsize=5, alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(cv_df['Model'], rotation=15, ha='right')
ax.set_ylabel('5-Fold CV Accuracy', fontsize=12)
ax.set_title('Cross-Validation Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0.6, 0.85])
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, cv_df['Mean Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning with GridSearchCV

Let's tune the **Random Forest** — the best general-purpose classifier.

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',  # Optimize for F1 since data is imbalanced
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

print(f"\nTest set results with tuned Random Forest:")
print(classification_report(y_test, y_pred_best,
                            target_names=['Non-diabetic', 'Diabetic']))

## 7. Effect of K on KNN

In [ ]:
k_range = range(1, 31)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, k_scores, 'o-', color='#F59E0B', markersize=5)
best_k = list(k_range)[np.argmax(k_scores)]
ax.axvline(best_k, color='#EF4444', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
ax.set_xlabel('K (Number of Neighbors)', fontsize=12)
ax.set_ylabel('Cross-Validation Accuracy', fontsize=12)
ax.set_title('KNN: Accuracy vs. K', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 Best K = {best_k} with accuracy = {max(k_scores):.4f}")
print("   Small K → complex boundary (overfit), Large K → smooth boundary (underfit)")

## ✅ Exercises for Students

1. **Try different SVM kernels**: Change `kernel='rbf'` to `'linear'` or `'poly'`. Which works best?
2. **Class imbalance**: Add `class_weight='balanced'` to Logistic Regression and Random Forest. Does recall for diabetic patients improve?
3. **Feature selection**: Try removing features with low correlation to Outcome. Does performance change?
4. **Clinical threshold**: For the best model, adjust the probability threshold from 0.5 to 0.3. What happens to Precision vs Recall?

```python
# Hint for Exercise 4:
# y_pred_custom = (best_model.predict_proba(X_test_scaled)[:, 1] >= 0.3).astype(int)
```